#Text Generation

In [ ]:
!pip install azure-core azure-ai-formrecognizer

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.ai.formrecognizer import AnalysisFeature
from collections import Counter

endpoint = "YOUR_API_ENDPOINT_HERE"
key = "YOUR_API_KEY_HERE"

local_file_path = "/content/giulia_rossi.png"
document_analysis_client = DocumentAnalysisClient(
    endpoint=endpoint, credential=AzureKeyCredential(key)
)

with open(local_file_path, "rb") as file:
    poller = document_analysis_client.begin_analyze_document("prebuilt-document", document=file, features=[
                AnalysisFeature.STYLE_FONT
            ])
    result = poller.result()

fonts = []

for style in result.styles:
    if style.similar_font_family:
        font = style.similar_font_family.split(",")[0].strip()
        fonts.append(font)

font_counts = Counter(fonts)
majority_font = font_counts.most_common(1)[0][0]

print("Majority Font:", majority_font)

bounding_box_values = {}
key_value_dict = {}
for kv_pair in result.key_value_pairs:
    key_content = None
    value_content = None
    if kv_pair.key:
        key_content = kv_pair.key.content
        key_bounding_box = kv_pair.key.bounding_regions[0].polygon if kv_pair.key.bounding_regions else None
        print(f"Key: '{key_content}'")
        print(f"  Key Bounding Box: {key_bounding_box}")

    if kv_pair.value:
        value_content = kv_pair.value.content
        value_bounding_box = kv_pair.value.bounding_regions[0].polygon if kv_pair.value.bounding_regions else None
        print(f"Value: '{value_content}'")
        print(f"  Value Bounding Box: {value_bounding_box}")
        bounding_box_values[value_content] = value_bounding_box
    if key_content and value_content:
        key_value_dict[key_content] = value_content

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

image = cv2.imread(local_file_path)

for kv_pair in result.key_value_pairs:
    if kv_pair.value and kv_pair.value.bounding_regions:
        bounding_box = kv_pair.value.bounding_regions[0].polygon
        if bounding_box:
            points = np.array([[int(point.x), int(point.y)] for point in bounding_box], np.int32)
            points = points.reshape((-1, 1, 2))

            cv2.polylines(image, [points], isClosed=True, color=(255, 255, 0), thickness=2)

    if kv_pair.key and kv_pair.key.bounding_regions:
        bounding_box = kv_pair.key.bounding_regions[0].polygon
        if bounding_box:
            points = np.array([[int(point.x), int(point.y)] for point in bounding_box], np.int32)
            points = points.reshape((-1, 1, 2))

            cv2.polylines(image, [points], isClosed=True, color=(255, 255, 0), thickness=2)


output_file_path = "/content/Image_with_bounding_boxes.png"
cv2.imwrite(output_file_path, image)
cv2_imshow(image)

In [ ]:
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

client = OpenAI(
    base_url="https://api.chatanywhere.tech/v1"
)

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."}
]

user_input = f"""
    I performed OCR recognition on an identity document (VISA) and extracted the following text:
    {result.key_value_pairs}
    For each field in the text, indicate whether it is static (unchangeable) or dynamic (changeable) in the context of generating multiple copies for different users.
    Please follow this format for the output:
    <output><text>Field Name:<s>Bounding Box of variable that can be associated to the field name from key_value_pairs (only provide the list of coordinates)</s><s>YES</s></text> (for static fields)
    <output><text>Field Name:<s>Bounding Box of variable that can be associated to the field name from key_value_pairs (only provide the list of coordinates)</s><s>NO</s></text> (for dynamic fields)
    Example: Field Name - document number, Bounding Box of Value : string comprising both alphabets and numericals.
    Important:
    1. Give the bounding box structure as it is given in the key_value_pairs, starting from polygon.
    2. Segregate dynamic and static fields efficiently like dynamic fields include user details (name, gender, dates etc.), type of the document they are holding, suppose for document number - value is the combination of numbers and alphabets.
    3. Please give it in the above format only, don't change anything, even <s> and </s>. If you find anything that is not valid, please remove it from the final output. You are very intelligent; only consider information pairs that are dynamic (i.e., can be change from one document to another).
    4. Don't leave out any valid key-value pair that is related to the user (like name, gender, place etc) and type of the document (like P, S and D for Passports, and type/class of Visa).
    """

messages.append({"role": "user", "content": user_input})

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=0.7
)

reply = response.choices[0].message.content
print(reply)

In [ ]:
import re
pattern = r"<text>(.*?)<s>(.*?)</s><s>(.*?)</s></text>"
matches = re.findall(pattern, reply)

In [ ]:
matches

In [ ]:

image = cv2.imread(local_file_path)
for tup in matches:
        bounding_box_str = tup[1]
        if bounding_box_str:
            points_str = re.findall(r"Point\(x=(\d+\.\d+), y=(\d+\.\d+)\)", bounding_box_str)
            points = np.array([[int(float(x)), int(float(y))] for x, y in points_str], np.int32)
            points = points.reshape((-1, 1, 2))
            cv2.polylines(image, [points], isClosed=True, color=(255, 0, 255), thickness=4)

output_file_path = "/content/dynamic_data_bounding_boxes.png"
cv2.imwrite(output_file_path, image)
cv2_imshow(image)

In [ ]:
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

client = OpenAI(
    base_url="https://api.chatanywhere.tech/v1"
)

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."}
]

user_input = f"""
    I performed OCR recognition on an identity document and extracted the following text:
    {matches}
    I am also providing you with the sample key-value dictionary, so as to give you an idea in which format (Like captial, data format, space) the dummy value should be generated. This the dictionary: {key_value_dict}.
    For each field in the text, generate a realistic dummy value corresponding to that field and return the output in the following format:
    <output><text>here data you generated:<s>here bounding box coordinates</s></text></output>
    <output><text>here full name of the identity holder that you have generated<s></s></text></output>
    <output><text>Gender of the user write M or F<s></s></text></output>
    Important:
    1. Only give the output in the above format.
    2. Only provide the generated data values; do not include the field names. If you do not know anything about the field, you can give the same value from the example dictionary, ***For Type/class and Entries give the exact same value from the dictionary also dont provide space in dates***.
    3. After all the dummy data and the coordinates, output the full name that you have generated along with the gender at the end.
    4. Follow the exact format above and do not change anything, including <s> and </s>.
    5. Generate different dummy data each time (avoid repeating similar values across iterations).
    6. The generated data must be in English.
    7. Process all elements in the provided text.
    """

messages.append({"role": "user", "content": user_input})

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=0.7
)

dummy = response.choices[0].message.content
print(dummy)

In [ ]:
pattern_values = r"<text>(.*?)<s>(.*?)</s></text>"
dummy_data = re.findall(pattern_values, dummy)

pattern_empty_values = r"<text>(.*?)<s></s></text>"
all_texts = re.findall(pattern_empty_values, dummy)

extra_fields = [t for t in all_texts if "<s>" not in t]
username = extra_fields[0]
gender = extra_fields[1]

In [ ]:
dummy_data

In [ ]:
username, gender

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from google.colab.patches import cv2_imshow

image_path = "/content/visa template.png"
image_pil = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image_pil)

def parse_points(point_string, row_i):
    clean_string = re.sub(r'Point\(x=([\d.]+), y=([\d.]+)\)', r'(\1, \2)', point_string)
    points = eval(clean_string)
    pad_x = 10
    pad_y = 20
    return [
        (int(float(x)+pad_x), int(float(y)-(-pad_y if row_i == 2 else 0)))
        for x, y in points
    ]

font_path = "/content/arial.ttf"
font = ImageFont.truetype(font_path, 30)

for row_i, (text, points_string) in enumerate(dummy_data[:-2]):
    points = parse_points(points_string, row_i)
    x1, y1 = points[0]
    draw.text((x1, y1), text, fill="black", font=font)

output_path = "synthetic_image.png"
image_pil.save(output_path)

cv2_imshow(cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR))

#Image Generation

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO("yolov8n.pt")

results = model("/content/giulia_rossi.png")

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

In [ ]:
def generate_person(gender_word, width, height):
    prompt = f"""
    realistic passport-size photo of a {gender_word},
    head and shoulders visible,
    chest up portrait,
    front facing,
    neutral expression,
    plain white background,
    wearing formals
    """

    negative_prompt = "close-up face, cropped shoulders, zoomed face, blurry, black and white image, colored bacground"

    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=1024,
        height=1024,
        num_inference_steps=15,
        guidance_scale=7.5
    ).images[0]

    image = np.array(image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    image = cv2.resize(image, (width, height))

    return image

In [ ]:
model = YOLO("yolov8n.pt")

def detect_passport_photo(image_path):
    img = cv2.imread(image_path)
    results = model(image_path)

    best_box = None
    best_conf = -1

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])

            if model.names[cls] == "person":
                if conf > best_conf:
                    best_conf = conf
                    best_box = box

    if best_box is not None:
        x1, y1, x2, y2 = map(int, best_box.xyxy[0])
        cropped = img[y1:y2, x1:x2]
        return img, cropped, (x1, y1, x2, y2)

    raise Exception("No person detected")

img, cropped, box = detect_passport_photo("/content/giulia_rossi.png")
x1, y1, x2, y2 = box
width = x2 - x1
height = y2 - y1

image = cv2.imread("/content/synthetic_image.png")
gender_word = "man" if gender == "M" else "woman"

generated_person = generate_person(gender_word, width, height)
generated_person = cv2.resize(generated_person, (width, height))

image[y1:y2, x1:x2] = generated_person
cv2.imwrite("final_output.jpg", image)
cv2_imshow(image)

#Signature Generation

In [ ]:
from transformers import pipeline

detector = pipeline(
    "object-detection",
    model="mdefrance/yolos-small-signature-detection"
)

In [ ]:
result = detector("/content/Fatima Ibrahim_test.png")
image = Image.open("/content/Fatima Ibrahim_test.png")

draw = ImageDraw.Draw(image)
box = result[-1]["box"]

draw.rectangle(
    [box["xmin"], box["ymin"], box["xmax"], box["ymax"]],
    outline="red",
    width=3
)
display(image)

In [ ]:
import random

def generate_signature(name, font_paths):
    font_path = random.choice(font_paths)
    font_size = 50

    font = ImageFont.truetype(font_path, font_size)
    img = Image.new("RGBA", (600,600), (255,255,255,0))
    draw = ImageDraw.Draw(img)

    ink = (
        random.randint(0,40),
        random.randint(0,40),
        random.randint(0,40),
        random.randint(180,255)
    )
    draw.text((10,10), name, fill=ink, font=font)
    angle = random.uniform(0,15)
    img = img.rotate(angle, expand=True)

    return img

In [ ]:
def paste_signature(doc_path, signature_img, box):
    doc = Image.open(doc_path).convert("RGBA")
    signature_img = signature_img.crop(signature_img.getbbox())

    xmin, ymin = box["xmin"], box["ymin"]
    xmax, ymax = box["xmax"], box["ymax"]
    pad_x, pad_y = 30,30
    box_w = xmax - xmin + pad_x
    box_h = ymax - ymin + pad_y

    sig_w, sig_h = signature_img.size

    scale = min(box_w / sig_w, box_h / sig_h)

    new_w = int(sig_w * scale)
    new_h = int(sig_h * scale)

    signature_img = signature_img.resize((new_w, new_h))
    x_offset = xmin + (box_w - new_w) // 2
    y_offset = ymin + (box_h - new_h) // 2
    doc.paste(signature_img, (x_offset, y_offset), signature_img)

    return doc

In [ ]:
font_paths = [
    "/content/modernline.otf",
    "/content/Cintarini.ttf",
]

name = username.split()[0]
box = result[-1]["box"]

signature = generate_signature(name, font_paths)
final_doc = paste_signature("/content/final_output.jpg", signature, box)

final_doc.save("document_with_signature.png")
display(final_doc)